# 04 — Agriculture Suitability Model

Builds the agricultural suitability surface from NDVI, land cover, slope, water proximity, road access, and settlement proximity.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import rasterio
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from utils import load_config, setup_logger
from fuzzy_standardisation import standardise_raster
from suitability_model import (weighted_linear_combination,
                                classify_suitability, build_constraint_mask)
from ahp import build_agriculture_ahp

logger = setup_logger("04_agriculture")
ag_cfg  = load_config(ROOT / "config/agriculture_config.yml")
INTERIM = ROOT / "data/interim"
PROC    = ROOT / "data/processed"
OUT_R   = ROOT / "outputs/rasters"
OUT_M   = ROOT / "outputs/maps"
PROC.mkdir(exist_ok=True)

print("Agriculture Suitability Model")
print(f"Criteria : {list(ag_cfg['criteria'].keys())}")
print(f"CR       : {ag_cfg['ahp_matrix']['consistency_ratio']}")


## Step 1 — Verify AHP Weights

In [ ]:
ahp = build_agriculture_ahp()
print(ahp.consistency_report())


## Step 2 — Land Cover Reclassification

In [ ]:
# Reclassify ESA WorldCover to agriculture suitability scores
import numpy as np
import rasterio
from rasterio.enums import Resampling

lc_path = INTERIM / "land_cover_30m_aligned.tif"
reclass_path = INTERIM / "land_cover_ag_reclass.tif"
reclass_map = ag_cfg["land_cover_reclass"]

if lc_path.exists():
    with rasterio.open(lc_path) as src:
        lc = src.read(1)
        profile = src.profile.copy()
        profile.update(dtype="float32", nodata=np.nan)

    reclass = np.full(lc.shape, np.nan, dtype="float32")
    for code, score in reclass_map.items():
        reclass = np.where(lc == int(code), float(score), reclass)

    with rasterio.open(reclass_path, "w", **profile) as dst:
        dst.write(reclass, 1)

    # Class distribution
    print("Land cover reclassification summary:")
    for code, score in sorted(reclass_map.items()):
        count = (lc == int(code)).sum()
        if count > 0:
            print(f"  Class {code:>3} -> score {score:.1f} | {count:>8,} pixels")
    print(f"\nReclassified raster saved: {reclass_path}")
else:
    print(f"Land cover raster not found: {lc_path}")


## Step 3 — Fuzzy Standardise All Criteria

In [ ]:
RASTER_MAP = {
    "ndvi":                   INTERIM / "ndvi_30m_aligned.tif",
    "land_cover_suitability": INTERIM / "land_cover_ag_reclass.tif",
    "slope":                  INTERIM / "slope_degrees_aligned.tif",
    "distance_to_water":      INTERIM / "dist_water_aligned.tif",
    "distance_to_roads":      INTERIM / "dist_roads_aligned.tif",
    "distance_to_settlements":INTERIM / "dist_settlements_aligned.tif",
}

std_rasters = {}
for name, cfg in ag_cfg["criteria"].items():
    src = RASTER_MAP.get(name)
    if src and src.exists():
        dst = PROC / f"ag_{name}_std.tif"
        standardise_raster(src, dst, cfg["fuzzy_type"], cfg["fuzzy_params"])
        std_rasters[name] = dst
    else:
        logger.warning(f"Input not found for: {name}")

print(f"\nStandardised {len(std_rasters)} / {len(ag_cfg['criteria'])} criteria")


## Step 4 — Constraint Mask

In [ ]:
slope_raster = INTERIM / "slope_degrees_aligned.tif"
lc_raster    = INTERIM / "land_cover_30m_aligned.tif"
ref = INTERIM / "dem_30m.tif"
mask_path = OUT_R / "agriculture_constraint_mask.tif"

constraint_inputs = {}
if slope_raster.exists():
    constraint_inputs["steep_slope"]  = (slope_raster, "gt", 15)   # exclude >15 deg
if lc_raster.exists():
    constraint_inputs["water_bodies"] = (lc_raster, "eq", 70)
    constraint_inputs["urban"]        = (lc_raster, "eq", 50)

if ref.exists() and constraint_inputs:
    build_constraint_mask(constraint_inputs, ref, mask_path)
    print(f"Constraint mask saved: {mask_path}")
else:
    mask_path = None


## Step 5 — WLC + Classify + Map

In [ ]:
weights   = {name: cfg["weight"] for name, cfg in ag_cfg["criteria"].items()}
suit_path = OUT_R / "agriculture_suitability.tif"
cls_path  = OUT_R / "agriculture_classified.tif"

if std_rasters:
    weighted_linear_combination(std_rasters, weights, suit_path, mask_path)
    classify_suitability(suit_path, cls_path)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    with rasterio.open(suit_path) as src:
        arr = np.where(src.read(1) == src.nodata, np.nan, src.read(1))
    im1 = axes[0].imshow(arr, cmap="YlGn", vmin=0, vmax=1)
    axes[0].set_title("Agriculture Suitability (continuous)", fontsize=11)
    axes[0].axis("off")
    plt.colorbar(im1, ax=axes[0], fraction=0.03, label="Suitability score")

    with rasterio.open(cls_path) as src:
        cls = src.read(1)
    cmap_cls = mcolors.ListedColormap(["#d73027","#fc8d59","#fee090","#91cf60","#1a9850"])
    norm_cls = mcolors.BoundaryNorm([0.5,1.5,2.5,3.5,4.5,5.5], 5)
    im2 = axes[1].imshow(cls, cmap=cmap_cls, norm=norm_cls)
    axes[1].set_title("Agriculture Suitability (classified)", fontsize=11)
    axes[1].axis("off")
    cbar2 = plt.colorbar(im2, ax=axes[1], fraction=0.03, ticks=[1,2,3,4,5])
    cbar2.set_ticklabels(["Very Low","Low","Moderate","High","Very High"])

    plt.suptitle("Agriculture Suitability — 19 Rural LGAs, Oyo State", fontsize=13)
    plt.tight_layout()
    fig.savefig(OUT_M / "agriculture_suitability_map.png", dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Map saved: {OUT_M / 'agriculture_suitability_map.png'}")
